In [8]:
from typing import Annotated
from dotenv import load_dotenv
load_dotenv()

from typing_extensions import TypedDict
from langchain.chat_models import init_chat_model
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages

In [ ]:
import os
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

from langchain.chat_models import init_chat_model

In [ ]:
llm = init_chat_model("groq:qwen/qwen3-32b")

class State(TypedDict):
    messages: Annotated[list, add_messages]

def chatbot(state: State) -> State:
    return {"messages": [llm.invoke(state["messages"])]}

builder = StateGraph(State)
builder.add_node("chatbot_node", chatbot)

builder.add_edge(START, "chatbot_node")
builder.add_edge("chatbot_node", END)

graph = builder.compile()

In [ ]:
message = {"role": "user", "content": "Who wrote the play 'Romeo and Juliet'? Print only the name"}
# message = {"role": "user", "content": "What is the latest price of MSFT stock?"}
response = graph.invoke({"messages":[message]})

response["messages"]

In [ ]:
state = None
while True:
    in_message = input("Human >> ")
    if in_message.lower() in {"bye", "done"}:
        break
    if state is None:
        state: State = {
            "messages": [{"role": "user", "content": in_message}]
        }
    else:
        state["messages"].append({"role": "user", "content": in_message})

    state = graph.invoke(state)
    print("Assistant ::", state["messages"][-1].content)

### 🤔 Before you move on
This chatbot can only answer from what the model already knows — it has no way to look up something real and current, like today's actual stock price. How would you give a graph node the ability to call an external function when it needs live data? That's `4_tool_call.ipynb`.